# S3 · Calidad y particionamiento de datos

**Integrante** · Joel Jonatan Quispe Mamani

**Dimensión** · Velocidad mediana en vuelo

Equipo 04 · AeroStreamPeru · Ciclo 9

Para la mediana y los percentiles usamos las velocidades válidas de los estados en vuelo. Medimos cuántos estados respaldan cada ventana. Una ventana sin velocidades válidas conserva sus percentiles nulos.

## 1. Bronze y alcance

El ejemplo usa el bloque del 1 al 5 de enero. La comprobación de esquema y tiempos nulos abarca el bloque completo. La limpieza detallada y las salidas Silver y Gold usan los instantes alineados cada dos minutos en UTC, una cadencia comparable con la fuente en vivo. `BLOQUES` permite ampliar el alcance.

In [1]:
integrante = "joel"
columna_principal = "velocidad_mediana_t"

from pathlib import Path

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import BooleanType, DoubleType, LongType, StringType, StructType

BRONZE = Path("/opt/s01-arquitectura/data/2026")
BLOQUES = ["01/2026-01-01_2026-01-05.parquet"]
CADENCIA_SEGUNDOS = 120
VELOCIDAD_MIN_M_S = 10
VELOCIDAD_MAX_M_S = 350
ARTIFACTS = Path("/opt/s03-procesamiento-calidad-datos/artifacts") / integrante
RUTA_SILVER = ARTIFACTS / "silver_estados"
RUTA_GOLD = ARTIFACTS / "gold_ventanas"

spark = (
    SparkSession.builder.master("local[2]")
    .appName(f"AeroStreamPeru-S03-{integrante}")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 15:39:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/24 15:39:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
esquema = (
    StructType()
    .add("time", LongType())
    .add("icao24", StringType())
    .add("lat", DoubleType())
    .add("lon", DoubleType())
    .add("onground", BooleanType())
    .add("velocity", DoubleType())
    .add("baroaltitude", DoubleType())
    .add("vertrate", DoubleType())
    .add("lastposupdate", DoubleType())
    .add("lastcontact", DoubleType())
)
rutas = [str(BRONZE / nombre) for nombre in BLOQUES]
tipos = {campo.name: campo.dataType for campo in spark.read.parquet(*rutas).schema}
incompatibles = [campo.name for campo in esquema if tipos.get(campo.name) != campo.dataType]
assert not incompatibles, f"Columnas con esquema incompatible: {incompatibles}"

bronze = spark.read.schema(esquema).parquet(*rutas)
bronze.printSchema()
bronze.agg(
    F.count("*").alias("filas_bronze"),
    (F.count("*") - F.count("time")).alias("tiempos_nulos"),
).show()

muestra = bronze.where(f"pmod(time, {CADENCIA_SEGUNDOS}) = 0").persist()
capturas_observadas = muestra.select("time").distinct()
muestra.orderBy("time", "icao24").show(5, truncate=False)

nulos_por_columna = [(F.count("*") - F.count(nombre)).alias(nombre) for nombre in muestra.columns]
auditoria = muestra.agg(F.count("*").alias("filas_muestra"), *nulos_por_columna).first()
print(f"Filas auditadas a 2 minutos: {auditoria.filas_muestra:,}")
for nombre in muestra.columns:
    print(f"{nombre}: {auditoria[nombre]:,} nulos")

root
 |-- time: long (nullable = true)
 |-- icao24: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- onground: boolean (nullable = true)
 |-- velocity: double (nullable = true)
 |-- baroaltitude: double (nullable = true)
 |-- vertrate: double (nullable = true)
 |-- lastposupdate: double (nullable = true)
 |-- lastcontact: double (nullable = true)



+------------+-------------+
|filas_bronze|tiempos_nulos|
+------------+-------------+
|     2365631|            0|
+------------+-------------+



+----------+------+-------------------+------------------+--------+------------------+------------+--------+----------------+----------------+
|time      |icao24|lat                |lon               |onground|velocity          |baroaltitude|vertrate|lastposupdate   |lastcontact     |
+----------+------+-------------------+------------------+--------+------------------+------------+--------+----------------+----------------+
|1767225720|0ac9f0|-2.6839599609375   |-68.77306404760327|false   |236.84601732833866|11262.36    |0.32512 |1.767225719403E9|1.76722571993E9 |
|1767225720|0aca5e|-1.76531982421875  |-72.75868044061176|false   |233.0743599518658 |11590.02    |0.32512 |1.767225654355E9|1.767225654355E9|
|1767225720|0c205c|-3.5155318955243615|-72.53562137998387|false   |239.7375276509851 |10668.0     |0.0     |1.767225719664E9|1.76722571993E9 |
|1767225720|0c2063|-4.350725594213486 |-72.91152427936424|false   |240.92022748534623|11277.6     |0.0     |1.767225719664E9|1.767225719664E9|

## 2. Calidad y escritura Silver

- `.na.drop()` exige tiempo, identificador, posición y estado en tierra. El filtro por cadencia ya excluye tiempos nulos, que se auditan antes de muestrear. Se normaliza `icao24` y se filtra por formato y rectángulo de estudio.
- Los campos opcionales conservan sus nulos. Convertimos NaN a nulo. La columna de velocidad válida toma los valores mayores que 10 y menores que 350 m/s. El intervalo es exploratorio y Silver conserva también la velocidad original.
- `dropDuplicates()` quita filas idénticas entre las columnas seleccionadas. Después, `row_number()` conserva una fila por `(icao24, time)`. La prioridad es el último contacto, la última posición y, para empates, el contenido de la fila.

Silver conserva los estados válidos en tierra y en vuelo. El filtro de vuelo se aplica a los indicadores.

In [3]:
obligatorias = ["time", "icao24", "lat", "lon", "onground"]
opcionales = ["velocity", "baroaltitude", "vertrate", "lastposupdate", "lastcontact"]

candidatos = (
    muestra.withColumn("icao24", F.lower(F.trim("icao24")))
    .withColumns({nombre: F.nanvl(F.col(nombre), F.lit(None).cast("double")) for nombre in opcionales})
    .na.drop(subset=obligatorias)
    .filter(
        F.col("icao24").rlike(r"^[0-9a-f]{6}$")
        & F.col("lat").between(-18.35, -0.04)
        & F.col("lon").between(-81.33, -68.65)
    )
    .persist()
)

filas_candidatas = candidatos.count()
sin_exactos = candidatos.dropDuplicates().persist()
filas_sin_exactos = sin_exactos.count()
prioridad = Window.partitionBy("icao24", "time").orderBy(
    F.col("lastcontact").desc_nulls_last(),
    F.col("lastposupdate").desc_nulls_last(),
    F.to_json(F.struct(*sin_exactos.columns)).asc(),
)
unicos = (
    sin_exactos.withColumn("prioridad", F.row_number().over(prioridad))
    .filter(F.col("prioridad") == 1)
    .drop("prioridad")
    .persist()
)
filas_unicas = unicos.count()
assert filas_unicas == unicos.select("icao24", "time").distinct().count()

print(f"Filas sin identidad, ubicación o estado válido: {auditoria.filas_muestra - filas_candidatas:,}")
print(f"Duplicados exactos eliminados: {filas_candidatas - filas_sin_exactos:,}")
print(f"Estados adicionales por clave eliminados: {filas_sin_exactos - filas_unicas:,}")

silver = unicos.withColumn("captura_utc", F.timestamp_seconds("time")).withColumn(
    "velocidad_valida_m_s",
    F.when(
        (F.col("velocity") > VELOCIDAD_MIN_M_S) & (F.col("velocity") < VELOCIDAD_MAX_M_S),
        F.col("velocity"),
    ),
)
silver.write.mode("overwrite").parquet(str(RUTA_SILVER))
silver = spark.read.parquet(str(RUTA_SILVER))
assert silver.count() == filas_unicas

en_vuelo = silver.filter(~F.col("onground"))
en_vuelo.agg(
    F.count("*").alias("estados_en_vuelo"),
    (F.count("*") - F.count("velocity")).alias("velocidad_sin_dato"),
    (F.count("velocity") - F.count("velocidad_valida_m_s")).alias("velocidad_fuera_de_rango"),
).show()

Filas sin identidad, ubicación o estado válido: 0
Duplicados exactos eliminados: 0
Estados adicionales por clave eliminados: 0


+----------------+------------------+------------------------+
|estados_en_vuelo|velocidad_sin_dato|velocidad_fuera_de_rango|
+----------------+------------------+------------------------+
|           17784|                10|                      13|
+----------------+------------------+------------------------+



## 3. Indicador Gold por ventana

Para la mediana y los percentiles usamos las velocidades válidas de los estados en vuelo. Medimos cuántos estados respaldan cada ventana. Una ventana sin velocidades válidas conserva sus percentiles nulos. Gold incluye percentiles, porcentaje de velocidades válidas y volumen promedio de contexto.

Una captura observada sin aeronaves elegibles aporta cero al conteo, con `.na.fill(0)`. Una captura ausente se registra como faltante dentro de su ventana. Cada ventana tiene cinco instantes esperados. Si no se observó ninguna captura, la ventana queda ausente.

In [4]:
conteos = en_vuelo.groupBy("time").agg(F.countDistinct("icao24").alias("aeronaves"))
capturas_con_volumen = (
    capturas_observadas.join(conteos, "time", "left")
    .na.fill(0, subset=["aeronaves"])
    .withColumn("captura_utc", F.timestamp_seconds("time"))
)
volumen = capturas_con_volumen.groupBy(
    F.window("captura_utc", "10 minutes").alias("ventana")
).agg(
    F.count("*").alias("capturas_observadas"),
    F.avg("aeronaves").alias("volumen_promedio_t"),
)

mediciones = (
    en_vuelo.groupBy(F.window("captura_utc", "10 minutes").alias("ventana"))
    .agg(
        F.count("*").alias("observaciones_en_vuelo"),
        F.count("velocidad_valida_m_s").alias("velocidades_validas"),
        F.percentile_approx("baroaltitude", 0.5).alias("altitud_mediana_t"),
        F.percentile_approx("velocidad_valida_m_s", [0.25, 0.50, 0.75]).alias("percentiles"),
        F.percentile_approx("vertrate", 0.5).alias("velocidad_vertical_mediana_t"),
    )
    .withColumn("velocidad_p25_t", F.get("percentiles", 0))
    .withColumn("velocidad_mediana_t", F.get("percentiles", 1))
    .withColumn("velocidad_p75_t", F.get("percentiles", 2))
    .drop("percentiles")
)

gold = (
    volumen.join(mediciones, "ventana", "left")
    .na.fill(0, subset=["observaciones_en_vuelo", "velocidades_validas"])
    .withColumn("inicio_utc", F.col("ventana.start"))
    .withColumn("fecha_utc", F.to_date("inicio_utc"))
    .withColumn("inicio_lima", F.from_utc_timestamp("inicio_utc", "America/Lima"))
    .withColumn("hora_del_dia", F.hour("inicio_lima"))
    .withColumn("dia_semana", F.dayofweek("inicio_lima"))
    .withColumn("capturas_esperadas", F.lit(600 // CADENCIA_SEGUNDOS))
    .withColumn("capturas_faltantes", F.col("capturas_esperadas") - F.col("capturas_observadas"))
    .drop("ventana", "inicio_lima")
    .withColumn(
        "porcentaje_velocidades_validas",
        F.when(
            F.col("observaciones_en_vuelo") > 0,
            100 * F.col("velocidades_validas") / F.col("observaciones_en_vuelo"),
        ),
    )
)

## 4. Escritura, lectura de vuelta y plan

Gold se escribe por `fecha_utc`, con un archivo por fecha en esta muestra. Tras leer la salida, conciliamos los conteos y revisamos el balance entre fechas. En `explain(True)`, `PartitionFilters` permite a Spark elegir la carpeta de la fecha consultada antes de leer archivos.

In [5]:
filas_antes = gold.count()
assert filas_antes > 0
(
    gold.repartition(8, "fecha_utc")
    .write.mode("overwrite")
    .partitionBy("fecha_utc")
    .parquet(str(RUTA_GOLD))
)
gold_guardado = spark.read.parquet(str(RUTA_GOLD))
filas_guardadas = gold_guardado.count()
assert filas_guardadas == filas_antes
print(f"Ventanas conciliadas en Gold: {filas_guardadas:,}")

gold_guardado.groupBy("fecha_utc").agg(
    F.count("*").alias("ventanas"),
    F.sum("capturas_faltantes").alias("capturas_faltantes_en_ventanas_observadas"),
    F.sum(F.when(F.col("capturas_faltantes") == 0, 1).otherwise(0)).alias("ventanas_completas"),
).sort("fecha_utc").show()
for carpeta in sorted(RUTA_GOLD.glob("fecha_utc=*")):
    archivos = len(list(carpeta.glob("*.parquet")))
    assert archivos == 1, (carpeta.name, archivos)
    print(f"{carpeta.name}: {archivos} archivo Parquet")

fecha_verificada = gold_guardado.agg(F.min("fecha_utc").alias("fecha")).first().fecha
particion = gold_guardado.filter(F.col("fecha_utc") == F.lit(fecha_verificada))
particion.explain(True)
particion.sort("inicio_utc").select(
    "inicio_utc", "capturas_observadas", "capturas_faltantes", columna_principal
).show(8)

Ventanas conciliadas en Gold: 588
+----------+--------+-----------------------------------------+------------------+
| fecha_utc|ventanas|capturas_faltantes_en_ventanas_observadas|ventanas_completas|
+----------+--------+-----------------------------------------+------------------+
|2026-01-01|     144|                                       24|               120|
|2026-01-02|     120|                                       20|               100|
|2026-01-03|      36|                                        9|                30|
|2026-01-04|     144|                                       24|               120|
|2026-01-05|     144|                                       24|               120|
+----------+--------+-----------------------------------------+------------------+

fecha_utc=2026-01-01: 1 archivo Parquet
fecha_utc=2026-01-02: 1 archivo Parquet
fecha_utc=2026-01-03: 1 archivo Parquet
fecha_utc=2026-01-04: 1 archivo Parquet
fecha_utc=2026-01-05: 1 archivo Parquet
== Parsed Logical 

## 5. Interpretación y defensa individual

**Bronze** es el Parquet de Trino. **Silver** contiene los estados normalizados y deduplicados de la muestra. **Gold** contiene el indicador por ventana y su cobertura.

Una medición de velocidad faltante reduce la cobertura. Mantener esa distinción evita que los consumidores confundan falta de información con una velocidad igual a cero. Verificar esquema, claves y nulos antes de publicar Gold evita que BI o ML tengan que reinterpretar registros erróneos después de consumirlos.

**Hallazgo del bloque de ejemplo.** En la muestra se encontraron 10 estados en vuelo sin velocidad y 13 con velocidad fuera del rango exploratorio. Esos 23 estados se excluyeron de los percentiles y se conservaron en el conteo de observaciones en vuelo para medir la cobertura.

Para el informe, guarda evidencia del esquema, los controles, la conciliación de conteos y `PartitionFilters`. La guía pide capturas con reloj y perfil visibles.

In [6]:
spark.stop()